In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

storage_account = "hospitalanalyticstorage"

# ADSL configuration
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(scope = "hospitalanalyticsvaultscope", key = "storage-connection")
)

bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/patient_flow"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/patient_flow"

# read from bronze
bronze_df = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
)

# Define Schema
schema = StructType([
    StructField("patient_id", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("admission_time", StringType(), True),
    StructField("discharge_time", StringType(), True),
    StructField("bed_id", IntegerType(), True),
    StructField("hospital_id", IntegerType(), True)
])

# Parse it to dataframe
parsed_df = bronze_df.withColumn("data", from_json(col("raw_json"), schema)).select("data.*")

# Convert type to Timestamp
clean_df = parsed_df.withColumn("admission_time", to_timestamp("admission_time"))
clean_df = clean_df.withColumn("discharge_time", to_timestamp("discharge_time"))

# Invalid admission_time
clean_df = clean_df.withColumn("admission_time",
                               when(
                                   (col("admission_time").isNull()) | (col("admission_time") > current_timestamp()),
                                   current_timestamp())
                               .otherwise(col("admission_time"))
                               )

# Handler Invalid Age
clean_df = clean_df.withColumn("age",
                               when(col("age") > 100, floor(rand() * 90+1).cast("int"))
                               .otherwise(col("age"))
                               )

# Schema Evolution
expected_cols = ["patient_id", "gender", "age", "department", "admission_time", "discharge_time", "bed_id", "hospital_id"]

for col_name in expected_cols:
    if col_name not in clean_df.columns:
        clean_df = clean_df.withColumn(col_name, lit(None))

(
    clean_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("mergeSchema", "true")
    .option("checkpointLocation", silver_path + "_checkpoint")
    .start(silver_path)
)

In [0]:
display(spark.read.format("delta").load(silver_path))

patient_id,gender,age,department,admission_time,discharge_time,bed_id,hospital_id
b0def3c8-c213-4be6-a8c9-d994741b9939,Female,13,Oncology,2026-05-26T21:06:15.994898Z,2026-05-28T07:06:15.994898Z,488,6
a0299672-58c2-41fe-9d27-fcec0e8b2d0f,Female,5,Surgery,2026-05-28T19:06:16.995686Z,2026-05-30T06:06:16.995686Z,361,7
88563d90-1f3a-4dae-a576-a32f6139b3ad,Male,46,Surgery,2026-05-27T06:07:15.060994Z,2026-05-30T03:07:15.060994Z,412,7
a8aefce7-46bc-4f8a-8696-cf1c84f5d50e,Male,30,Pediatrics,2026-05-27T11:07:16.061837Z,2026-05-28T14:07:16.061837Z,288,7
ec90efbe-2ac6-4649-a43e-1c726e5fb174,Female,38,ICU,2026-05-29T07:07:20.066582Z,2026-05-30T23:07:20.066582Z,259,6
2d7a5f0a-467e-4c83-9628-98d6846b36a7,Male,1,Emergency,2026-05-26T14:07:21.067632Z,2026-05-27T05:07:21.067632Z,49,1
5bcea11e-7bd3-472b-b7a2-ef6b74b62d28,Female,44,ICU,2026-05-28T04:07:47.096729Z,2026-05-29T06:07:47.096729Z,309,4
f072cff6-d365-4b48-ab34-adb26f02b62b,Male,56,ICU,2026-05-27T01:07:48.098035Z,2026-05-29T14:07:48.098035Z,416,3
937fb033-c22d-4221-b031-f2a28174aa94,Female,72,Pediatrics,2026-05-28T06:54:41.162888Z,2026-05-30T17:54:41.162888Z,175,1
5b7ba666-5733-4424-9374-1e2ddce399fb,Male,89,ICU,2026-05-28T15:54:42.163728Z,2026-05-29T10:54:42.163728Z,247,4
